<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="../images/nvidia.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building LLM Applications With Prompt Engineering</b></font></h1>
<h2><b>Structured Output</b></h2>
<br>


## License

Unless otherwise noted:

- Code cells are licensed under the
  [Apache License 2.0](../../../LICENSES/Apache-2.0.txt).
- Markdown and raw cells are licensed under the
  [Creative Commons Attribution 4.0 International License](../../../LICENSES/CC-BY-4.0.txt).
- Third-party material retains its original license and is identified in
  [`THIRD_PARTY_NOTICES.md`](../../../THIRD_PARTY_NOTICES.md).

Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES.

In this notebook you will use the model's strong JSON generation as a starting point, then inspect the more useful question: whether the JSON is dependable enough for downstream code.

```mermaid
flowchart LR
    A(["Model output"]):::pill
    B["Looks like JSON<br>syntax is clean"]:::warn
    C{"Contract boundary"}:::diamond
    D["Schema<br>fields + types"]:::box
    E["Field policy<br>allowed / required"]:::box
    F["Missing values<br>null, omit, default"]:::box
    G(["Usable structured object"]):::pill_end

    A --> B --> C
    C --> D --> G
    C --> E --> G
    C --> F --> G

    classDef pill fill:#1a3a2a,stroke:#3fb950,stroke-width:2px,color:#3fb950,rx:20
    classDef pill_end fill:#0d2b40,stroke:#58a6ff,stroke-width:2px,color:#58a6ff,rx:20
    classDef box fill:#161b22,stroke:#30363d,stroke-width:1.5px,color:#e6edf3
    classDef diamond fill:#1f1a2e,stroke:#8b5cf6,stroke-width:2px,color:#c4b5fd
    classDef warn fill:#2b2111,stroke:#f59e0b,stroke-width:2px,color:#fbbf24
```

---

## Objectives

By the time you complete this notebook you will:

- Prompt the model to generate structured JSON.
- Parse model output into Python objects.
- Check whether valid JSON also satisfies a minimal application contract.
- Use the chat model to batch process inputs into structured data.

---

## Imports

In [ ]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, SimpleJsonOutputParser
from langchain_core.runnables import RunnableLambda
from IPython.display import display, JSON
import json

---

## Create a Model Instance

### API key for this notebook kernel

Paste your NVIDIA API key when the next cell prompts for it. The typed value is visible during entry but is not placed in the cell source or written to a file. It remains only in this kernel's memory. Restarting the kernel discards it, and a separate notebook kernel will ask for the key again.

Avoid entering the key while screen sharing or recording.

In [ ]:
import os

if not os.environ.get("NVIDIA_API_KEY"):
    _nvidia_api_key = input("Paste your NVIDIA API key for this notebook: ").strip()
    if not _nvidia_api_key:
        raise ValueError("An NVIDIA API key is required to run this notebook.")
    os.environ["NVIDIA_API_KEY"] = _nvidia_api_key
    del _nvidia_api_key

os.environ.setdefault("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
print("NVIDIA API access is configured for this notebook kernel.")

In [ ]:
base_url = os.getenv("NVIDIA_BASE_URL")
model = 'nvidia/nemotron-3-super-120b-a12b'
llm = ChatNVIDIA(
    base_url=base_url, model=model, temperature=0, max_completion_tokens=1024,
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)

---

## LLMs and Highly Structured Data Formats

A very common task we would like LLMs to perform is to generate outputs in a highly structured format. These formats could be as common as JSON, or a Python list, or some custom structure unique to our needs like a custom report or document structure, just to name a few examples.

Modern models are often quite good at this. In fact, the model used here can usually produce valid JSON from a simple instruction. That changes the lesson a bit: the limitation is less often *can the model write braces correctly?* and more often *did it create the exact object your application needs?*

Let's start with JSON, then add small checks that separate syntactic success from a reliable contract.

---

## A Simple JSON Object

Start with a small set of supplied facts about Santa Clara and ask the model to represent them as JSON. Keep those facts fixed while refining the field and formatting instructions, so you can inspect structure without relying on recalled population figures or other unsupplied details.

In [ ]:
city_facts = "Santa Clara is a city in California, United States. It is in Silicon Valley."
prompt = f"Make a JSON object representing the city Santa Clara. Use only these supplied facts: {city_facts}"

In [ ]:
result = llm.invoke(prompt).content
print(result)

Check that the output uses only the supplied facts. Then check whether its structure meets the assumptions of downstream code:

In [ ]:
def check_contract(result: str | dict, required_fields: set):
    parsed_json = result
    valid_json = True
    if isinstance(result, str):
        try:
            parsed_json = json.loads(result)
        except json.JSONDecodeError:
            valid_json = False
            parsed_json = None

    valid_object = isinstance(parsed_json, dict)
    observed_fields = set(parsed_json) if valid_object else set()
    return {
        "valid_json": valid_json,
        "valid_object": valid_object,
        "missing_fields": sorted(required_fields - observed_fields),
        "extra_fields": sorted(observed_fields - required_fields),
    }

check_contract(result, {"name", "country", "fun_facts"})

You may get back clean JSON immediately. That is good news, but it is not the finish line.

Look at the contract check above. The object can parse successfully and still be awkward for downstream code: the model may choose a field like `city` instead of `name`, add useful-but-unrequested fields, or omit a field the next component expects. The next prompt narrows the request so the model has less freedom to invent the record shape. However, it still could fail in theory...


In [ ]:
prompt = f'''\
Make a JSON object representing the city Santa Clara. \
Use only these supplied facts: {city_facts}
It should have fields for:
- The name of the city.
- The country the city is located in.
- Short fun facts about the city.
Only return the JSON block. Never return non-JSON text.'''

In [ ]:
result = llm.invoke(prompt).content
print(result)
check_contract(result, {"name", "country", "fun_facts"})

This is getting closer. One more instruction makes the output safer for parsers if a model decides to wrap the JSON in Markdown backticks.


In [ ]:
prompt = f'''\
Make a JSON object representing the city Santa Clara. \
Use only these supplied facts: {city_facts}
It should have fields for:
- The name of the city
- The country the city is located in.
- Short fun facts about the city.
Only return the JSON. Never return non-JSON text, including backtick wrappers around the JSON.'''

In [ ]:
result = llm.invoke(prompt).content
print(result)
check_contract(result, {"name", "country", "fun_facts"})

And hopefully, there you go (and if not, there is probably a small amount of extra code or prompt notes you could bring in to make it work)

---

## Make a Template Out of the Prompt

Parameterize both the city name and its supplied facts. Keep the same evidence boundary as we turn the prompt into a reusable template.

In [ ]:
json_city_template = ChatPromptTemplate.from_template('''\
Make a JSON object representing the city {city_name}. \
Use only these supplied facts: {city_facts}
It should have fields for:
- The name of the city
- The country the city is located in.
- Short fun facts about the city.
Only return the JSON. Never return non-JSON text, including backtick wrappers around the JSON.''')

Next we'll compose a simple chain.

In [ ]:
parser = StrOutputParser()

In [ ]:
chain = json_city_template | llm | parser

In [ ]:
print(chain.invoke({'city_name': 'Santa Clara', 'city_facts': city_facts}))

This also looks good.

---

## Simple JSON Parsing

To confirm that we can load the JSON object as a Python dict, we can use a custom runnable to parse the model response directly to a Python dict.

In [ ]:
parse_to_dict = RunnableLambda(lambda response: json.loads(response.content))

We'll re-compose our chain to use this custom parser.

In [ ]:
# chain = json_city_template | llm | parser | json.loads  ## Equivalent
chain = json_city_template | llm | parse_to_dict

In [ ]:
chain.invoke({'city_name': 'Santa Clara', 'city_facts': city_facts})

Successful parsing establishes valid JSON. It does not establish whether the object contains the right fields, uses stable field names, or contains facts suitable for your application.

LangChain provides `SimpleJsonOutputParser` for this parsing step. Reconstruct the chain using it, while keeping those remaining checks in mind.

In [ ]:
from langchain_core.output_parsers import SimpleJsonOutputParser

In [ ]:
json_parser = SimpleJsonOutputParser()

In [ ]:
chain = json_city_template | llm | json_parser

In [ ]:
chain.invoke({'city_name': 'Santa Clara', 'city_facts': city_facts})

---

## Batch on Multiple Inputs

So far so good, but continuing in the spirit of iterative prompt development, now let's try our chain on several different inputs.

In [ ]:
city_inputs = [
    {'city_name': 'Santa Clara', 'city_facts': city_facts},
    {'city_name': 'New Orleans', 'city_facts': 'New Orleans is in Louisiana, United States. It lies on the Mississippi River.'},
    {'city_name': 'Busan', 'city_facts': 'Busan is a port city in South Korea. It lies on the southeast coast.'},
    {'city_name': 'Cairo', 'city_facts': 'Cairo is the capital of Egypt. It lies on the Nile River.'},
    {'city_name': 'Perth', 'city_facts': 'Perth is the capital of Western Australia, Australia. It lies on the Swan River.'}
]

In [ ]:
city_details = chain.batch(city_inputs, config={"max_concurrency": 1})

Now check the outputs as application records, not just JSON blobs. This tiny report asks whether every row has the fields the next step expects.


In [ ]:
required_city_fields = {"name", "country", "fun_facts"}
city_contract_report = []

for city_name, row in zip(city_inputs, city_details):
    city_contract_report.append(check_contract(row, required_city_fields))

city_contract_report

In [ ]:
city_details

In [ ]:
for city in city_details:
    contract = check_contract(city, {"name", "country", "fun_facts"})
    if not contract["valid_object"] or contract["missing_fields"]:
        print("[CONTRACT FAILURE]", contract)
        continue
    print("\nCity:", city["name"])
    print("Country:", city["country"])
    print("Fun Facts:")
    for fact in city["fun_facts"]:
        print(f" - {fact}")

---

## Structure and Generation

Compare each output with its input facts as well as the field contract. The model has transformed supplied text into structured records; it should not add population figures, rankings or other facts from memory. Valid JSON and correct field names still do not prove that every value is supported.

---

## Exercise: Generate a List of Book Details

Using the techniques you've learned thus far, generate a python list containing dictionaries that each contain details about the following books.

Each dict should have the book's title, author, and year of original publication.

Feel free to check out the *Solution* below if you get stuck.

In [ ]:
sci_fi_books = [
    {"book_title": "Dune"},
    {"book_title": "Neuromancer"},
    {"book_title": "Snow Crash"},
    {"book_title": "The Left Hand of Darkness"},
    {"book_title": "Foundation"}
]

### Your Work Here

### Solution

In [ ]:
book_template = ChatPromptTemplate.from_template('''\
Make a JSON object representing the details of the following book: {book_title}. \
It should have fields for:
- The title of the book.
- The author of the book.
- The year the book was originally published.

Only return the JSON. Never return non-JSON text, including backtick wrappers around the JSON.''')

In [ ]:
chain = book_template | llm | json_parser

In [ ]:
chain.batch(sci_fi_books, config={"max_concurrency": 1})

---

## Summary

In this notebook you began approaching the technique of LLMs generating structured output. In the next notebook you're going to drastically increase your capabilities in this arena by using Pydantic classes and LangChain's JsonOutputParser.

---

<!-- NEXT_STEP_CARD -->

<div style="border-left: 6px solid #76B900; background: #f7fdf2; padding: 14px 18px; border-radius: 10px; margin: 20px 0;">
<p style="margin: 0 0 6px; color: #315c00; font-weight: 700; letter-spacing: .04em; text-transform: uppercase;">Continue the unified course path</p>
<p style="margin: 0 0 8px; color: black;"><strong>Next step:</strong> Open <code>4-Structured-Output/42-Pydantic.ipynb</code> next: <strong>Structured Output with Pydantic</strong>.</p>
<p style="margin: 0; color: black;"><strong>Before moving on:</strong> Keep one output schema you care about, so the next notebook can show how typed validation tightens the contract around it.</p>
</div>